In [2]:
import torch
import torch.nn as nn
 
# ──────────────────────────────────────────────
# 1. The XOR dataset (all 4 points)
# ──────────────────────────────────────────────
X_xor = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y_xor = torch.tensor([[0],    [1],    [1],    [0]],    dtype=torch.float32)
 
 
# ──────────────────────────────────────────────
# 2. Check linear separability
# ──────────────────────────────────────────────
 
def is_linearly_separable(X: torch.Tensor, y: torch.Tensor,
                          lr: float = 0.1, epochs: int = 1000,
                          threshold: float = 1.0) -> dict:
    """
    Try to learn a linear decision boundary.
    If final accuracy < 100%, the data is NOT linearly separable.
 
    Returns a dict with the verdict, final accuracy, and learned params.
    """
    n_features = X.shape[1]
 
    # Pure linear model: Linear → Sigmoid (no hidden layer)
    model = nn.Sequential(nn.Linear(n_features, 1), nn.Sigmoid())
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
 
    # Train
    for _ in range(epochs):
        loss = criterion(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
    # Evaluate
    with torch.no_grad():
        preds = (model(X) >= 0.5).float()
        accuracy = (preds == y).float().mean().item()
 
    w = model[0].weight.data.squeeze().tolist()
    b = model[0].bias.data.item()
 
    return {
        "separable":   accuracy >= threshold,
        "accuracy":    accuracy,
        "final_loss":  loss.item(),
        "weights":     w,
        "bias":        b,
    }
 
 
# ──────────────────────────────────────────────
# 3. Solve XOR with a hidden layer
# ──────────────────────────────────────────────
 
def solve_xor(X: torch.Tensor, y: torch.Tensor,
              lr: float = 0.5, epochs: int = 2000) -> dict:
    """
    Solve XOR using a 2-layer net (2 → 4 → 1).
    This works because the hidden layer makes it linearly separable
    in a higher-dimensional space.
    """
    model = nn.Sequential(
        nn.Linear(2, 4),   # hidden layer lifts into 4-D
        nn.ReLU(),
        nn.Linear(4, 1),
        nn.Sigmoid(),
    )
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
 
    for _ in range(epochs):
        loss = criterion(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
    with torch.no_grad():
        preds = (model(X) >= 0.5).float()
        accuracy = (preds == y).float().mean().item()
 
    return {"accuracy": accuracy, "final_loss": loss.item(), "model": model}
 
 
# ──────────────────────────────────────────────
# 4. Example usage
# ──────────────────────────────────────────────
 
if __name__ == "__main__":
 
    print("=" * 50)
    print("  XOR Linear Separability Check")
    print("=" * 50)
 
    # ── Test 1: XOR (should FAIL) ──
    print("\n── Dataset: XOR ──")
    print("  (0,0)→0   (0,1)→1   (1,0)→1   (1,1)→0\n")
 
    result = is_linearly_separable(X_xor, y_xor)
    print(f"  Linearly separable?  {'YES' if result['separable'] else 'NO'}")
    print(f"  Best accuracy:       {result['accuracy']:.0%}")
    print(f"  Final loss:          {result['final_loss']:.4f}")
    print(f"  Learned weights:     {result['weights']}")
    print(f"  Learned bias:        {result['bias']:.4f}")
 
    # ── Test 2: AND (should PASS) ──
    print("\n── Dataset: AND ──")
    X_and = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
    y_and = torch.tensor([[0],    [0],    [0],    [1]],    dtype=torch.float32)
 
    result = is_linearly_separable(X_and, y_and)
    print(f"  Linearly separable?  {'YES' if result['separable'] else 'NO'}")
    print(f"  Best accuracy:       {result['accuracy']:.0%}")
 
    # ── Test 3: OR (should PASS) ──
    print("\n── Dataset: OR ──")
    X_or = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
    y_or = torch.tensor([[0],    [1],    [1],    [1]],    dtype=torch.float32)
 
    result = is_linearly_separable(X_or, y_or)
    print(f"  Linearly separable?  {'YES' if result['separable'] else 'NO'}")
    print(f"  Best accuracy:       {result['accuracy']:.0%}")
 
    # ── Solve XOR with a hidden layer ──
    print("\n" + "=" * 50)
    print("  Solving XOR with a hidden layer (2 → 4 → 1)")
    print("=" * 50)
 
    sol = solve_xor(X_xor, y_xor)
    print(f"\n  Accuracy:    {sol['accuracy']:.0%}")
    print(f"  Final loss:  {sol['final_loss']:.4f}")
 
    # Show predictions
    with torch.no_grad():
        raw = sol["model"](X_xor).squeeze()
    print("\n  Input   Target   Prediction")
    for i in range(4):
        x0, x1 = X_xor[i].tolist()
        t = int(y_xor[i].item())
        p = raw[i].item()
        print(f"  ({int(x0)},{int(x1)})    {t}        {p:.4f}  →  {int(p >= 0.5)}")
 
    # ── Why it works ──
    print("""
── Why XOR is not linearly separable ──
 
  Class 0: (0,0) and (1,1)     ← diagonally opposite
  Class 1: (0,1) and (1,0)     ← diagonally opposite
 
  No single straight line can separate them.
  A hidden layer transforms the space so a line CAN separate them.
""")


  XOR Linear Separability Check

── Dataset: XOR ──
  (0,0)→0   (0,1)→1   (1,0)→1   (1,1)→0

  Linearly separable?  NO
  Best accuracy:       75%
  Final loss:          0.6932
  Learned weights:     [-0.009129480458796024, -0.009147478267550468]
  Learned bias:        0.0108

── Dataset: AND ──
  Linearly separable?  YES
  Best accuracy:       100%

── Dataset: OR ──
  Linearly separable?  YES
  Best accuracy:       100%

  Solving XOR with a hidden layer (2 → 4 → 1)

  Accuracy:    75%
  Final loss:  0.4774

  Input   Target   Prediction
  (0,0)    0        0.3333  →  0
  (0,1)    1        0.3333  →  0
  (1,0)    1        1.0000  →  1
  (1,1)    0        0.3333  →  0

── Why XOR is not linearly separable ──

  Class 0: (0,0) and (1,1)     ← diagonally opposite
  Class 1: (0,1) and (1,0)     ← diagonally opposite

  No single straight line can separate them.
  A hidden layer transforms the space so a line CAN separate them.

